In [2]:
import os, sys, zipfile, subprocess
from pathlib import Path
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])

PROJECT = Path('/content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_resize_256')
DATA = Path('/content/iharmony4_256')
DATA.mkdir(parents=True, exist_ok=True)

ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']
for subset in ARCHIVES:
    dest = DATA / subset
    if not (dest / '.extracted_ok').exists():
        zip_path = PROJECT / f'{subset}_256.zip'
        assert zip_path.is_file(), f'Không tìm thấy {zip_path}'
        print(f'Đang giải nén {zip_path.name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA)
        (dest / '.extracted_ok').touch()

print('Dữ liệu 256x256 sẵn sàng tại:', DATA)

Đang giải nén HCOCO_256.zip...
Đang giải nén HAdobe5k_256.zip...
Đang giải nén HFlickr_256.zip...
Đang giải nén Hday2night_256.zip...
Dữ liệu 256x256 sẵn sàng tại: /content/iharmony4_256


In [6]:
import os
import glob
import shutil

# ================= SỬA ĐƯỜNG DẪN THƯ MỤC iharmony4_256 CỦA BẠN =================
DATASET_ROOT = "/content/iharmony4_256"  # Thay bằng đường dẫn thực tế (vd: /content/drive/MyDrive/iharmony4_256)
# ==============================================================================

REPO_DIR = "/content/HarmonyTransformer"
EXP_NAME = "HT_2H9L_allihd"

# 1. Tự động tạo file IHD_test.txt tổng ở thư mục gốc từ 4 file *_test.txt con
subsets = ["HCOCO", "HAdobe5k", "HFlickr", "Hday2night"]
ihd_test_path = os.path.join(DATASET_ROOT, "IHD_test.txt")

with open(ihd_test_path, "w") as out_f:
    total_lines = 0
    for sub in subsets:
        sub_txt = os.path.join(DATASET_ROOT, sub, f"{sub}_test.txt")
        if os.path.exists(sub_txt):
            with open(sub_txt, "r") as in_f:
                for line in in_f:
                    line = line.strip()
                    if not line:
                        continue
                    # Nếu dòng chưa có tiền tố tên thư mục con (vd: HCOCO/...), tự động thêm vào
                    if not line.startswith(sub + "/"):
                        line = f"{sub}/{line}"
                    out_f.write(line + "\n")
                    total_lines += 1

print(f"Đã tạo xong {ihd_test_path} với tổng cộng {total_lines} mẫu test.")

# 2. Clone repo & cài thư viện
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/zhenglab/HarmonyTransformer.git {REPO_DIR}
!pip install -q gdown dominate visdom

# 3. Tải checkpoint pre-trained chính chủ từ Google Drive
os.makedirs("/content/ht_download", exist_ok=True)
downloaded_file = "/content/ht_download/pretrained_ht.zip"
if not os.path.exists(downloaded_file):
    !gdown --fuzzy "https://drive.google.com/file/d/1uQqveBSUfTmvA4FEWC_stAyf2oMS4UHC/view?usp=sharing" -O {downloaded_file}

with open(downloaded_file, "rb") as f:
    magic = f.read(4)

if magic[:2] == b"PK":
    !unzip -q -o {downloaded_file} -d /content/ht_download/extracted
    pth_files = glob.glob("/content/ht_download/extracted/**/*.pth", recursive=True)
    ht_pth = [p for p in pth_files if "HT_2H9L" in p and "DHT" not in p]
    CKPT_PATH = ht_pth[0] if ht_pth else pth_files[0]
else:
    CKPT_PATH = downloaded_file

ckpt_dir = os.path.join(REPO_DIR, "checkpoints", EXP_NAME)
os.makedirs(ckpt_dir, exist_ok=True)
shutil.copy(CKPT_PATH, os.path.join(ckpt_dir, "latest_net_G.pth"))
print("Đã nạp checkpoint:", CKPT_PATH)

Đã tạo xong /content/iharmony4_256/IHD_test.txt với tổng cộng 7404 mẫu test.
Đã nạp checkpoint: /content/ht_download/extracted/HT_DHT_pretrained/FCHT_2H9L_allihd/latest_net_G.pth


In [9]:
import os

DATASET_ROOT = "/content/iharmony4_256"
subsets = ["HCOCO", "HAdobe5k", "HFlickr", "Hday2night"]

# Ghi lại IHD_test.txt đúng chuẩn: chỉ "HCOCO/ten_anh.jpg"
ihd_test_path = os.path.join(DATASET_ROOT, "IHD_test.txt")
with open(ihd_test_path, "w") as out_f:
    total = 0
    for sub in subsets:
        sub_txt = os.path.join(DATASET_ROOT, sub, f"{sub}_test.txt")
        if os.path.exists(sub_txt):
            with open(sub_txt, "r") as in_f:
                for line in in_f:
                    line = line.strip()
                    if not line:
                        continue
                    filename = os.path.basename(line)
                    out_f.write(f"{sub}/{filename}\n")
                    total += 1

print(f"Đã sửa {ihd_test_path} ({total} ảnh) theo đúng chuẩn!")

# Chạy test.py trên toàn bộ 7,404 ảnh (--num_test 7404)
%cd /content/HarmonyTransformer
!CUDA_VISIBLE_DEVICES=0 python test.py \
  --model ht \
  --tr_r_enc_head 2 \
  --tr_r_enc_layers 9 \
  --name HT_2H9L_allihd \
  --dataset_root /content/iharmony4_256/ \
  --dataset_name IHD \
  --batch_size 12 \
  --num_threads 2 \
  --num_test 7404 \
  --init_port 8097 \
  --eval

Đã sửa /content/iharmony4_256/IHD_test.txt (7404 ảnh) theo đúng chuẩn!
/content/HarmonyTransformer
----------------- Options ---------------
             DIST_BACKEND: nccl                          
               NUM_SHARDS: 1                             
                 RNG_SEED: 1                             
                 SHARD_ID: 0                             
                    activ: lrelu                         
             aspect_ratio: 1.0                           
               batch_size: 12                            	[default: 4]
          checkpoints_dir: ./checkpoints                 
                crop_size: 256                           
             dataset_mode: ihd                           
             dataset_name: IHD                           
             dataset_root: /content/iharmony4_256/       	[default: ../../datasets/ihd_datasets/all/]
              dim_forward: 2                             
                direction: AtoB                 

In [15]:
!cat /content/iharmony4_256/HCOCO_test.txt /content/iharmony4_256/HAdobe5k_test.txt /content/iharmony4_256/HFlickr_test.txt /content/iharmony4_256/Hday2night_test.txt > /content/iharmony4_256/IHD_test.txt
%cd /content/HarmonyTransformer
!CUDA_VISIBLE_DEVICES=0 python evaluation/ih_evaluation.py --dataroot /content/iharmony4_256/ --result_root results/HT_2H9L_allihd/test_latest/images/ --evaluation_type our --dataset_name ALL

/content/HarmonyTransformer
HCOCO
100% 4283/4283 [00:36<00:00, 116.53it/s]
4283
HCOCO MSE 21.63 | PSNR 37.29 | SSIM 0.9762 |fMSE 358.57 | fPSNR 24.34 | fSSIM 0.8868
HAdobe5k
100% 2160/2160 [00:17<00:00, 121.88it/s]
2160
HAdobe5k MSE 48.81 | PSNR 35.56 | SSIM 0.9288 |fMSE 336.54 | fPSNR 25.62 | fSSIM 0.8361
HFlickr
100% 828/828 [00:07<00:00, 111.86it/s]
828
HFlickr MSE 83.24 | PSNR 32.02 | SSIM 0.9572 |fMSE 607.38 | fPSNR 22.13 | fSSIM 0.8555
Hday2night
100% 133/133 [00:01<00:00, 128.62it/s]
133
Hday2night MSE 51.84 | PSNR 36.05 | SSIM 0.9223 |fMSE 796.31 | fPSNR 21.67 | fSSIM 0.6631
IHD
100% 7404/7404 [01:02<00:00, 118.45it/s]
7404
IHD MSE 36.99 | PSNR 36.17 | SSIM 0.9593 |fMSE 387.83 | fPSNR 24.42 | fSSIM 0.8645


In [16]:
import os
import shutil
from google.colab import drive

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Tạo thư mục lưu trữ trên Drive
SAVE_DIR = "/content/drive/MyDrive/ImageHarmonization/runs/"
os.makedirs(SAVE_DIR, exist_ok=True)

# 3. Lưu file checkpoint pre-trained (nặng ~19MB - copy trong 1 giây)
ckpt_src = "/content/HarmonyTransformer/checkpoints/HT_2H9L_allihd/latest_net_G.pth"
if os.path.exists(ckpt_src):
    shutil.copy(ckpt_src, os.path.join(SAVE_DIR, "HT_2H9L_allihd_latest_net_G.pth"))
    print("Đã lưu checkpoint vào:", os.path.join(SAVE_DIR, "HT_2H9L_allihd_latest_net_G.pth"))

# 4. Nén toàn bộ thư mục kết quả (ảnh + file log điểm) thành 1 file .zip rồi mới chuyển sang Drive
results_src = "/content/HarmonyTransformer/results/HT_2H9L_allihd"
if os.path.exists(results_src):
    zip_path = "/content/HT_2H9L_results"
    shutil.make_archive(zip_path, 'zip', results_src)
    shutil.move(zip_path + ".zip", os.path.join(SAVE_DIR, "HT_2H9L_results.zip"))
    print("Đã nén và lưu toàn bộ kết quả vào:", os.path.join(SAVE_DIR, "HT_2H9L_results.zip"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã lưu checkpoint vào: /content/drive/MyDrive/ImageHarmonization/runs/HT_2H9L_allihd_latest_net_G.pth
Đã nén và lưu toàn bộ kết quả vào: /content/drive/MyDrive/ImageHarmonization/runs/HT_2H9L_results.zip
